# LeetCode 522 — Longest Uncommon Subsequence II bằng Python

Link đề: https://leetcode.com/problems/longest-uncommon-subsequence-ii/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Two Pointers, String, Sorting

Skeleton LeetCode:

```python
class Solution:
    def findLUSlength(self, strs: List[str]) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: trong danh sách chuỗi `strs`, tìm **độ dài lớn nhất** của một chuỗi là **dãy con của đúng một** chuỗi trong danh sách (không là dãy con của chuỗi nào khác); không có thì trả `-1`.
2. Nắm các công cụ Python: phân biệt **dãy con (subsequence)** với **chuỗi con (substring)**, kiểm tra dãy con bằng **hai con trỏ**, vòng `for ... else`, duyệt mọi tập con bằng **bitmask** (`1 << k`, `>>`, `&`), `Counter`, `sorted(key=len, reverse=True)`, `any`/`all`.
3. Đi từ **liệt kê mọi dãy con rồi kiểm tra** → **đếm tần suất mọi dãy con bằng hash** → **chỉ thử chính các chuỗi gốc** → **sắp xếp giảm dần theo độ dài + loại trùng + dừng sớm**.
4. Hiểu bước đột phá: nếu tồn tại một dãy con "không chung" thì **chính chuỗi chứa nó cũng "không chung"** và dài hơn ⇒ chỉ cần thử **các chuỗi nguyên bản**, không phải `2^m` dãy con.
5. Tránh các lỗi kinh điển: so sánh nội dung (`s != t`) thay vì chỉ số (`i != j`) làm hỏng trường hợp chuỗi trùng, nhầm dãy con với chuỗi con, chỉ đếm trùng mà quên kiểm tra "bị chứa", và kiểm tra ngược chiều.

## 2. Đề bài & ví dụ

Cho mảng chuỗi `strs`. Trả về **độ dài của dãy con không chung dài nhất** giữa chúng; nếu không tồn tại, trả `-1`.

- **Dãy con không chung (uncommon subsequence)** của mảng chuỗi là một chuỗi **là dãy con của đúng một chuỗi** và **không là dãy con của các chuỗi còn lại**.
- **Dãy con (subsequence)** của chuỗi `s` là chuỗi nhận được khi **xoá một số ký tự bất kỳ** (có thể không xoá, có thể xoá hết) mà **giữ nguyên thứ tự** các ký tự còn lại. Ví dụ `"abc"` là dãy con của `"aebdc"` (xoá `e`, `d`); `""` (chuỗi rỗng) là dãy con của mọi chuỗi.

**Ví dụ 1:** `strs = ["aba","cdc","eae"]` ⇒ `3`.
Giải thích: `"aba"` chỉ là dãy con của chính nó (`"cdc"` và `"eae"` không chứa `a-b-a` theo thứ tự) ⇒ độ dài 3.

**Ví dụ 2:** `strs = ["aaa","aaa","aa"]` ⇒ `-1`.
Giải thích: `"aaa"` xuất hiện hai lần (mỗi bản là dãy con của bản kia) và `"aa"` là dãy con của `"aaa"` ⇒ không có dãy con nào "độc quyền".

**Ràng buộc:**

- `2 <= strs.length <= 50`
- `1 <= strs[i].length <= 10`
- `strs[i]` gồm chữ cái thường tiếng Anh.

Cell dưới chạy thử input/output của hai ví dụ để làm quen với dạng dữ liệu (chưa giải).

In [1]:
from collections import Counter
from itertools import combinations
import random
import time
from typing import List

examples = [(["aba", "cdc", "eae"], 3), (["aaa", "aaa", "aa"], -1)]
for strs, expected in examples:
    print(strs, "-> mong đợi", expected, "| số chuỗi:", len(strs), "| độ dài lớn nhất:", max(map(len, strs)))

['aba', 'cdc', 'eae'] -> mong đợi 3 | số chuỗi: 3 | độ dài lớn nhất: 3
['aaa', 'aaa', 'aa'] -> mong đợi -1 | số chuỗi: 3 | độ dài lớn nhất: 3


## 3. Kiến thức nền

### 3.1. Dãy con (subsequence) khác chuỗi con (substring)

- **Chuỗi con** phải là một đoạn **liền nhau**: các chuỗi con của `"abc"` là `"a","b","c","ab","bc","abc"` (và `""`).
- **Dãy con** chỉ cần **giữ thứ tự**: ngoài các chuỗi con còn có `"ac"` (bỏ `b`). Số dãy con (theo cách chọn vị trí) của chuỗi dài `m` là `2^m`.

Toán tử `in` của Python kiểm tra **chuỗi con**, không phải dãy con.

In [2]:
print("ac" in "abc")        # False: 'ac' không phải chuỗi con (không liền nhau)
print("bc" in "abc")        # True
# 'ac' là dãy con của 'abc' (bỏ b) -> ta phải tự viết hàm kiểm tra, ở mục 3.2

False
True


### 3.2. Kiểm tra dãy con bằng hai con trỏ

Để kiểm tra `s` có phải dãy con của `t` hay không: dùng `i` trỏ vào `s`, `j` trỏ vào `t`.

- Duyệt `j` từ trái sang phải trên `t`.
- Nếu `t[j] == s[i]` thì "khớp được ký tự `s[i]`" ⇒ `i += 1`.
- Cuối cùng `s` là dãy con của `t` **khi và chỉ khi `i == len(s)`** (đã khớp hết `s`).

Đây là cách **tham lam**: gặp ký tự cần tìm là lấy ngay, vì lấy sớm nhất không bao giờ làm mất cơ hội khớp các ký tự sau.

In [3]:
def is_subseq(s, t):
    """True nếu s là dãy con của t."""
    i = 0
    for ch in t:                      # duyệt j trên t
        if i < len(s) and ch == s[i]:
            i += 1
    return i == len(s)


print(is_subseq("ace", "abcde"))   # True  (a, c, e theo thứ tự)
print(is_subseq("aec", "abcde"))   # False (e đứng trước c trong s nhưng sau c trong t)
print(is_subseq("", "abc"))        # True  (chuỗi rỗng là dãy con của mọi chuỗi)
print(is_subseq("abc", "ab"))      # False (s dài hơn t thì không thể)

True
False
True
False


### 3.3. `for ... else` trong Python

Khối `else` của vòng `for` chạy **chỉ khi vòng lặp kết thúc tự nhiên (không gặp `break`)**. Rất hợp với kiểu "kiểm tra không có phần tử nào thoả điều kiện": nếu gặp một phần tử thoả thì `break` (và bỏ qua `else`), không gặp thì chạy `else`.

In [4]:
nums = [3, 5, 7]
for x in nums:
    if x % 2 == 0:
        print("có số chẵn", x)
        break
else:
    print("không có số chẵn nào")      # in ra vì không có break

không có số chẵn nào


### 3.4. Liệt kê mọi dãy con bằng bitmask

Với chuỗi `s` dài `m`, mỗi số `mask` trong `0 .. 2^m - 1` mã hoá một lựa chọn: bit thứ `k` của `mask` bằng `1` nghĩa là **giữ** ký tự `s[k]`. Biểu thức `(mask >> k) & 1` lấy bit thứ `k`; `1 << m` là `2^m`.

In [5]:
def subsequences_bitmask(s):
    m = len(s)
    out = []
    for mask in range(1 << m):
        out.append("".join(s[k] for k in range(m) if (mask >> k) & 1))
    return out


print(subsequences_bitmask("abc"))     # 8 dãy con: '', 'a', 'b', 'ab', 'c', 'ac', 'bc', 'abc'
print(len(subsequences_bitmask("aaa")), "dãy con theo vị trí nhưng chỉ", len(set(subsequences_bitmask("aaa"))), "chuỗi khác nhau")

['', 'a', 'b', 'ab', 'c', 'ac', 'bc', 'abc']
8 dãy con theo vị trí nhưng chỉ 4 chuỗi khác nhau


### 3.5. Xây dần tập các dãy con khác nhau (bằng `set`)

Cách khác (không bitmask): bắt đầu với `{""}`; với mỗi ký tự `ch` của `s`, mỗi dãy con cũ `x` sinh thêm `x + ch` (chọn thêm ký tự này). `set` tự loại trùng, nên `"aaa"` chỉ còn 4 dãy con khác nhau: `"", a, aa, aaa`.

In [6]:
def subsequence_set(s):
    subs = {""}
    for ch in s:
        subs |= {x + ch for x in subs}      # giữ cũ (không chọn ch) + thêm bản có ch
    return subs


print(sorted(subsequence_set("abc")))
print(sorted(subsequence_set("aaa")))
assert subsequence_set("abca") == set(subsequences_bitmask("abca"))

['', 'a', 'ab', 'abc', 'ac', 'b', 'bc', 'c']
['', 'a', 'aa', 'aaa']


### 3.6. `Counter`, `sorted(key=len, reverse=True)`, `any`/`all`

- `Counter(list)` đếm số lần xuất hiện của mỗi phần tử.
- `sorted(strs, key=len, reverse=True)` sắp xếp theo **độ dài giảm dần**.
- `any(cond for x in ...)` là `True` nếu **có** phần tử thoả; `all(...)` là `True` nếu **mọi** phần tử thoả.

In [7]:
strs = ["aba", "cdc", "aba", "e"]
print(Counter(strs))
print(sorted(strs, key=len, reverse=True))
print(any(is_subseq("a", t) for t in strs), all(len(t) >= 2 for t in strs))

Counter({'aba': 2, 'cdc': 1, 'e': 1})
['aba', 'cdc', 'aba', 'e']
True False


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hiểu đề bằng ví dụ nhỏ:

**Ví dụ A:** `["aba","cdc","eae"]`. Ba chuỗi dùng bộ chữ khác nhau (gần như); `"aba"` có là dãy con của `"cdc"` không? `cdc` không có chữ `a` ⇒ không. Của `"eae"`? `eae` có một `a` nhưng cần `a-b-a` ⇒ không. Vậy `"aba"` độc quyền, và nó dài 3 (bằng độ dài tối đa) ⇒ đáp án 3.

**Ví dụ B:** `["aaa","aaa","aa"]`.
- `"aaa"` có xuất hiện ở chuỗi thứ hai ⇒ là dãy con của chuỗi khác ⇒ **không** độc quyền (chuỗi trùng nhau luôn chứa nhau).
- `"aa"` là dãy con của `"aaa"` ⇒ không độc quyền.
- Mọi dãy con của `"aaa"` hoặc `"aa"` đều có dạng `a...a` và nằm trong `"aaa"` kia ⇒ không còn gì độc quyền ⇒ `-1`.

**Ví dụ C:** `["abc","abc","ab"]` ⇒ `"abc"` trùng ⇒ loại; `"ab"` là dãy con của `"abc"` ⇒ loại ⇒ `-1`.

**Ví dụ D:** `["abc","ab","a"]` ⇒ `"abc"` không là dãy con của chuỗi ngắn hơn ⇒ độc quyền ⇒ `3`.

Quan sát trực giác: **chuỗi dài hơn có nhiều "cơ hội" độc quyền hơn** (khó bị chứa trong chuỗi khác). Chuỗi ngắn chỉ có thể bị chứa trong chuỗi **dài hơn hoặc bằng** nó — và nếu bằng nhau thì phải **giống hệt**.

## 5. Cách 1 — Brute Force / baseline: liệt kê mọi dãy con rồi kiểm tra

**Ý tưởng bám sát đề:** với mỗi chuỗi `strs[i]`, liệt kê **mọi dãy con** `sub` của nó (bitmask). `sub` là "không chung" nếu **không** là dãy con của bất kỳ `strs[j]`, `j != i`. Lấy độ dài lớn nhất.

Ta cố ý dùng một cách kiểm tra dãy con **khác** hai con trỏ (dùng `iter` — thủ thuật "mỗi ký tự của `s` phải xuất hiện lần lượt trong phần còn lại của `t`") để baseline này làm **oracle độc lập** khi đối chiếu ở mục 12.

Thủ thuật `iter`: `it = iter(t)`; `ch in it` duyệt `it` đến khi gặp `ch` (và **tiêu thụ** phần đã duyệt). `all(ch in it for ch in s)` thành đúng khi các ký tự của `s` lần lượt xuất hiện theo thứ tự trong `t`.

In [8]:
def is_subseq_iter(s, t):
    """Cách kiểm tra dãy con khác: dùng iterator của t (mỗi `ch in it` tiêu thụ it)."""
    it = iter(t)
    return all(ch in it for ch in s)


print(is_subseq_iter("ace", "abcde"), is_subseq_iter("aec", "abcde"), is_subseq_iter("", "x"))


def lus_bruteforce(strs):
    """Cách 1: mọi dãy con của mọi chuỗi, kiểm tra với các chuỗi còn lại."""
    best = -1
    for i, s in enumerate(strs):
        for sub in set(subsequences_bitmask(s)):
            if len(sub) <= best:
                continue                                   # không cải thiện đáp án
            if not any(is_subseq_iter(sub, t) for j, t in enumerate(strs) if j != i):
                best = len(sub)
    return best


for strs, expected in examples:
    print(strs, "->", lus_bruteforce(strs), "(mong đợi", expected, ")")
    assert lus_bruteforce(strs) == expected
print(lus_bruteforce(["abc", "ab", "a"]), lus_bruteforce(["abc", "abc", "ab"]))   # 3 và -1

True False True
['aba', 'cdc', 'eae'] -> 3 (mong đợi 3 )
['aaa', 'aaa', 'aa'] -> -1 (mong đợi -1 )
3 -1


**Độ phức tạp:** đặt `n = len(strs)`, `m` = độ dài lớn nhất. Mỗi chuỗi có tối đa `2^m` dãy con, mỗi dãy con kiểm tra với `n-1` chuỗi, mỗi lần `O(m)` ⇒ **`O(n² · 2^m · m)`** thời gian; bộ nhớ `O(2^m · m)`. Với `n = 50`, `m = 10`: khoảng `50·1024·50·10 ≈ 25 triệu` phép so — vẫn chạy được nhưng **chậm và không hề cần thiết**.

**Nút thắt:** lũy thừa `2^m` đến từ việc xét **mọi dãy con**. Liệu có cần xét cả chúng không? Nếu một dãy con ngắn là độc quyền thì chuỗi dài hơn chứa nó có thể cũng độc quyền — ta sẽ chứng minh ở mục 6.

## 6. Ý tưởng thuật toán / bước đột phá

**Bổ đề:** nếu tồn tại một dãy con không chung `u` (là dãy con của **đúng một** chuỗi `s = strs[i]`, không là dãy con của các chuỗi khác), thì **chính `s` cũng là một dãy con không chung**, và `len(s) ≥ len(u)`.

**Chứng minh (phản chứng):** giả sử `s` là dãy con của một chuỗi khác `t = strs[j]`, `j ≠ i`. Vì `u` là dãy con của `s`, mà `s` là dãy con của `t`, nên `u` cũng là dãy con của `t` (tính **bắc cầu** của "là dãy con"). Mâu thuẫn với việc `u` không là dãy con của chuỗi nào khác. Vậy `s` không là dãy con của chuỗi nào khác. Hơn nữa `u` là dãy con của `s` nên `len(u) ≤ len(s)`. ∎

**Hệ quả:** đáp án = **max** của `len(s)` trên các chuỗi `s = strs[i]` mà `s` **không là dãy con của bất kỳ `strs[j]` (`j ≠ i`)**. Chỉ còn `n` ứng viên thay vì `n · 2^m`!

(Hệ quả phụ: chuỗi trùng nhau — hai chỉ số khác nhau cùng nội dung — luôn "chứa nhau", nên cả hai bị loại; ta phải so sánh theo **chỉ số** `i != j`, không theo nội dung.)

Ta sẽ lần lượt xem: Cách 2 (đếm bằng hash, vẫn mũ), Cách 3 (ứng viên = chuỗi gốc, hai con trỏ — **code nộp**), Cách 4 (sắp xếp + loại trùng + dừng sớm).

## 7. Các cách tiếp theo

Tổng cộng 4 cách:

- Cách 1: mọi dãy con + kiểm tra (mục 5).
- Cách 2: đếm tần suất mọi dãy con bằng `Counter` (hash).
- Cách 3: chỉ thử các chuỗi gốc, kiểm tra hai con trỏ (**chính thức**).
- Cách 4: sắp xếp giảm dần theo độ dài + loại chuỗi trùng + dừng ngay khi tìm thấy.

---

### Cách 2 — Đếm tần suất mọi dãy con bằng `Counter`

Giải thích từng đoạn:

1. Với mỗi chuỗi `s`, dùng `subsequence_set(s)` (mục 3.5) lấy **tập các dãy con khác nhau** (loại trùng **bên trong** một chuỗi).
2. Với mỗi dãy con trong tập đó, `cnt[sub] += 1`. Vì mỗi chuỗi đóng góp **tối đa 1** cho mỗi `sub`, `cnt[sub]` = **số chuỗi (theo chỉ số) chứa `sub` làm dãy con**.
3. `sub` là "không chung" ⇔ `cnt[sub] == 1`.
4. Đáp án: `max(len(sub))` trên các `sub` có `cnt == 1`, hoặc `-1`.

Hai chuỗi trùng nội dung tự nhiên đóng góp hai lần cho mọi dãy con của chúng ⇒ `cnt ≥ 2` ⇒ bị loại. Không cần xử lý trùng riêng.

In [9]:
def lus_counter(strs):
    """Cách 2: đếm xem mỗi dãy con xuất hiện trong bao nhiêu chuỗi."""
    cnt = Counter()
    for s in strs:
        for sub in subsequence_set(s):       # mỗi chuỗi chỉ đóng góp 1 lần cho mỗi dãy con
            cnt[sub] += 1
    best = -1
    for sub, c in cnt.items():
        if c == 1:
            best = max(best, len(sub))
    return best


for strs, expected in examples:
    print(strs, "->", lus_counter(strs))
    assert lus_counter(strs) == expected

['aba', 'cdc', 'eae'] -> 3
['aaa', 'aaa', 'aa'] -> -1


**Code nộp LeetCode cho Cách 2:**

```python
from collections import Counter

class Solution:
    def findLUSlength(self, strs: List[str]) -> int:
        cnt = Counter()
        for s in strs:
            subs = {""}
            for ch in s:
                subs |= {x + ch for x in subs}
            for sub in subs:
                cnt[sub] += 1
        return max((len(sub) for sub, c in cnt.items() if c == 1), default=-1)
```

**Độ phức tạp:** thời gian `O(n · 2^m · m)` (mỗi chuỗi sinh tối đa `2^m` dãy con dài tới `m`), bộ nhớ `O(n · 2^m · m)` trong `Counter`. Nhanh hơn Cách 1 (bỏ hẳn thừa số `n` thứ hai) nhưng vẫn **mũ theo `m`** — được nhờ `m ≤ 10`; nếu chuỗi dài hơn thì sụp đổ. Không cần cho `m` lớn: xem Cách 3.

---

### Cách 3 — Chỉ thử các chuỗi gốc (chính thức)

Giải thích từng đoạn (dựa vào bổ đề mục 6):

1. Viết `check(s, t)`: `s` có là dãy con của `t` không (hai con trỏ, mục 3.2).
2. Với mỗi chỉ số `i`, xét `s = strs[i]`: duyệt mọi `j` khác `i`; nếu `check(s, strs[j])` đúng ⇒ `s` bị chứa ⇒ `break`.
3. Nếu vòng trong **không `break`** (nhờ `for ... else`) ⇒ `s` độc quyền ⇒ `ans = max(ans, len(s))`.
4. `ans` khởi tạo `-1`.

In [10]:
def lus_candidates(strs):
    """Cách 3: mỗi chuỗi gốc là một ứng viên; loại nếu là dãy con của chuỗi khác (so theo CHỈ SỐ)."""
    def check(s, t):
        i = j = 0
        while i < len(s) and j < len(t):
            if s[i] == t[j]:
                i += 1
            j += 1
        return i == len(s)

    ans = -1
    for i, s in enumerate(strs):
        for j, t in enumerate(strs):
            if i != j and check(s, t):
                break                       # s nằm trong chuỗi khác -> loại
        else:
            ans = max(ans, len(s))          # không bị chứa -> độc quyền
    return ans


for strs, expected in examples:
    print(strs, "->", lus_candidates(strs))
    assert lus_candidates(strs) == expected
print(lus_candidates(["aabbcc", "aabbcc", "cb"]))                      # 2
print(lus_candidates(["aabbcc", "aabbcc", "bc", "bcc", "aabbccc"]))    # 7

['aba', 'cdc', 'eae'] -> 3
['aaa', 'aaa', 'aa'] -> -1
2
7


**Code nộp LeetCode cho Cách 3:** (đó là code ở mục 9.)

**Độ phức tạp:** `n²` cặp `(i, j)`, mỗi cặp `check` `O(m)` ⇒ thời gian **`O(n² · m)`** (≈ `50·50·10 = 25 000` phép — rất nhỏ); bộ nhớ `O(1)` phụ.

---

### Cách 4 — Sắp xếp giảm dần + loại trùng + dừng sớm

Giải thích từng đoạn:

1. Sắp xếp `order = sorted(strs, key=len, reverse=True)` — chuỗi dài đứng trước.
2. Chuỗi `s` ở vị trí `i` chỉ có thể bị chứa trong chuỗi **dài hơn** (đứng **trước** nó) hoặc chuỗi **cùng độ dài** (khi đó phải **giống hệt**). Chuỗi đứng sau không thể chứa `s` (ngắn hơn hoặc bằng nhưng khác).
3. Loại những chuỗi **có bản sao** (`Counter[s] > 1`): chúng luôn bị chứa bởi bản sao. Chỉ xét `s` có `cnt[s] == 1`.
4. Với `s` như vậy, chỉ cần kiểm tra `s` với `order[:i]` (các chuỗi đứng trước). Nếu **không** chuỗi nào chứa `s` ⇒ vì ta duyệt **từ dài xuống ngắn**, đây chính là **đáp án** ⇒ `return len(s)` ngay (dừng sớm).
5. Duyệt hết mà không tìm được ⇒ `-1`.

In [11]:
def lus_sorted(strs):
    """Cách 4: sắp xếp giảm dần theo độ dài; ứng viên đầu tiên (không trùng, không bị chứa) là đáp án."""
    cnt = Counter(strs)
    order = sorted(strs, key=len, reverse=True)
    for i, s in enumerate(order):
        if cnt[s] > 1:
            continue                                           # có bản sao -> bị chứa
        if not any(is_subseq(s, t) for t in order[:i]):        # chỉ so với chuỗi dài hơn/bằng
            return len(s)                                      # dài nhất còn lại -> đáp án
    return -1


for strs, expected in examples:
    print(strs, "->", lus_sorted(strs))
    assert lus_sorted(strs) == expected
print(lus_sorted(["aabbcc", "aabbcc", "bc", "bcc", "aabbccc"]))    # 7

['aba', 'cdc', 'eae'] -> 3
['aaa', 'aaa', 'aa'] -> -1
7


**Code nộp LeetCode cho Cách 4:**

```python
from collections import Counter

class Solution:
    def findLUSlength(self, strs: List[str]) -> int:
        def check(s: str, t: str) -> bool:
            i = 0
            for ch in t:
                if i < len(s) and ch == s[i]:
                    i += 1
            return i == len(s)

        cnt = Counter(strs)
        order = sorted(strs, key=len, reverse=True)
        for i, s in enumerate(order):
            if cnt[s] == 1 and not any(check(s, t) for t in order[:i]):
                return len(s)
        return -1
```

**Độ phức tạp:** sắp xếp `O(n log n · m)`; vòng chính tệ nhất `O(n² · m)` nhưng **dừng sớm** ngay khi gặp ứng viên đầu tiên; bộ nhớ `O(n)` cho `order` và `Counter`. Thực tế thường nhanh hơn Cách 3 khi có chuỗi dài độc quyền ở đầu.

## 8. TRACE từng bước

### 8.1. Trace hàm `check(s, t)` (hai con trỏ)

Kiểm tra `s = "abc"` có là dãy con của `t = "aebdc"` không:

| bước | j | `t[j]` | `s[i]` | khớp? | i sau |
|---|---|---|---|---|---|
| 1 | 0 | a | a | có | 1 |
| 2 | 1 | e | b | không | 1 |
| 3 | 2 | b | b | có | 2 |
| 4 | 3 | d | c | không | 2 |
| 5 | 4 | c | c | có | 3 |

Kết thúc `i = 3 = len(s)` ⇒ `True`. Hàm trace bên dưới in đúng các dòng này, và một ca trả `False`.

In [12]:
def trace_check(s, t):
    i = 0
    print(f"kiểm tra s={s!r} là dãy con của t={t!r}")
    for j, ch in enumerate(t):
        want = s[i] if i < len(s) else None
        match = want is not None and ch == want
        if match:
            i += 1
        print(f"  j={j} t[j]={ch!r} đang cần s[i]={want!r} -> {'khớp' if match else 'không'}  i={i}")
    ok = i == len(s)
    print("  =>", ok, f"(i={i}, len(s)={len(s)})")
    return ok


assert trace_check("abc", "aebdc") is True
assert trace_check("aec", "abcde") is False

kiểm tra s='abc' là dãy con của t='aebdc'
  j=0 t[j]='a' đang cần s[i]='a' -> khớp  i=1
  j=1 t[j]='e' đang cần s[i]='b' -> không  i=1
  j=2 t[j]='b' đang cần s[i]='b' -> khớp  i=2
  j=3 t[j]='d' đang cần s[i]='c' -> không  i=2
  j=4 t[j]='c' đang cần s[i]='c' -> khớp  i=3
  => True (i=3, len(s)=3)
kiểm tra s='aec' là dãy con của t='abcde'
  j=0 t[j]='a' đang cần s[i]='a' -> khớp  i=1
  j=1 t[j]='b' đang cần s[i]='e' -> không  i=1
  j=2 t[j]='c' đang cần s[i]='e' -> không  i=1
  j=3 t[j]='d' đang cần s[i]='e' -> không  i=1
  j=4 t[j]='e' đang cần s[i]='e' -> khớp  i=2
  => False (i=2, len(s)=3)


### 8.2. Trace toàn bài (Cách 3) với `strs = ["aabbcc", "aabbcc", "cb"]`

Đáp án mong đợi là `2` (dãy con `"cb"`).

| i | `s` | so với j=0 | so với j=1 | so với j=2 | kết luận | ans |
|---|---|---|---|---|---|---|
| 0 | `aabbcc` | (bỏ, i=j) | `aabbcc` chứa nó ⇒ **break** | — | bị chứa | -1 |
| 1 | `aabbcc` | `aabbcc` chứa nó ⇒ **break** | (bỏ) | — | bị chứa | -1 |
| 2 | `cb` | `cb` có trong `aabbcc`? c sau mọi b ⇒ không | không | (bỏ) | không `break` ⇒ độc quyền | **2** |

Hai chuỗi trùng bị loại nhờ so theo **chỉ số** (`i != j`), còn `"cb"` ngắn nhưng độc quyền. Hàm `trace_lus` bên dưới in từng bước.

In [13]:
def trace_lus(strs):
    ans = -1
    for i, s in enumerate(strs):
        print(f"--- i={i} s={s!r}")
        contained_in = None
        for j, t in enumerate(strs):
            if i == j:
                print(f"    j={j}: bỏ qua (chính nó)")
                continue
            res = is_subseq(s, t)
            print(f"    j={j} t={t!r}: {s!r} là dãy con? {res}")
            if res:
                contained_in = j
                break
        if contained_in is None:
            ans = max(ans, len(s))
            print(f"    => {s!r} ĐỘC QUYỀN, ans = {ans}")
        else:
            print(f"    => {s!r} bị chứa trong strs[{contained_in}], loại")
    return ans


print("đáp án:", trace_lus(["aabbcc", "aabbcc", "cb"]))

--- i=0 s='aabbcc'
    j=0: bỏ qua (chính nó)
    j=1 t='aabbcc': 'aabbcc' là dãy con? True
    => 'aabbcc' bị chứa trong strs[1], loại
--- i=1 s='aabbcc'
    j=0 t='aabbcc': 'aabbcc' là dãy con? True
    => 'aabbcc' bị chứa trong strs[0], loại
--- i=2 s='cb'
    j=0 t='aabbcc': 'cb' là dãy con? False
    j=1 t='aabbcc': 'cb' là dãy con? False
    j=2: bỏ qua (chính nó)
    => 'cb' ĐỘC QUYỀN, ans = 2
đáp án: 2


### 8.3. Trace Cách 4 (sắp xếp) với `strs = ["aabbcc", "aabbcc", "bc", "bcc", "aabbccc"]`

`order = ["aabbccc", "aabbcc", "aabbcc", "bcc", "bc"]` (giảm dần theo độ dài). `Counter`: `aabbcc` xuất hiện 2 lần.

- `s = "aabbccc"` (đếm 1): so với `order[:0] = []` ⇒ không ai chứa ⇒ **return 7**. Dừng ngay, không cần xét các chuỗi sau.

In [14]:
def trace_sorted(strs):
    cnt = Counter(strs)
    order = sorted(strs, key=len, reverse=True)
    print("order =", order, "| đếm =", dict(cnt))
    for i, s in enumerate(order):
        if cnt[s] > 1:
            print(f"i={i} s={s!r}: có bản sao -> loại")
            continue
        holders = [t for t in order[:i] if is_subseq(s, t)]
        print(f"i={i} s={s!r}: chuỗi đứng trước chứa nó = {holders}")
        if not holders:
            print("   => không ai chứa, trả", len(s))
            return len(s)
    return -1


assert trace_sorted(["aabbcc", "aabbcc", "bc", "bcc", "aabbccc"]) == 7
print("----")
assert trace_sorted(["aaa", "aaa", "aa"]) == -1

order = ['aabbccc', 'aabbcc', 'aabbcc', 'bcc', 'bc'] | đếm = {'aabbcc': 2, 'bc': 1, 'bcc': 1, 'aabbccc': 1}
i=0 s='aabbccc': chuỗi đứng trước chứa nó = []
   => không ai chứa, trả 7
----
order = ['aaa', 'aaa', 'aa'] | đếm = {'aaa': 2, 'aa': 1}
i=0 s='aaa': có bản sao -> loại
i=1 s='aaa': có bản sao -> loại
i=2 s='aa': chuỗi đứng trước chứa nó = ['aaa', 'aaa']


## 9. Code nộp LeetCode

Cách 3 (chính thức): đúng `class Solution`, phương thức `findLUSlength(self, strs: List[str]) -> int`.

In [15]:
class Solution:
    def findLUSlength(self, strs: List[str]) -> int:
        def check(s: str, t: str) -> bool:
            i = j = 0
            while i < len(s) and j < len(t):
                if s[i] == t[j]:
                    i += 1
                j += 1
            return i == len(s)

        ans = -1
        for i, s in enumerate(strs):
            for j, t in enumerate(strs):
                if i != j and check(s, t):
                    break
            else:
                ans = max(ans, len(s))
        return ans


sol = Solution()
print(sol.findLUSlength(["aba", "cdc", "eae"]))    # 3
print(sol.findLUSlength(["aaa", "aaa", "aa"]))     # -1

3
-1


## 10. Vì sao đúng

Gọi `U` là tập chuỗi gốc **không là dãy con của bất kỳ chuỗi khác (theo chỉ số)**. Ta cần chứng minh: **đáp án đề bài = `max len(s)` với `s ∈ U`** (hoặc `-1` nếu `U` rỗng).

1. **(≥) Mỗi `s ∈ U` là một dãy con không chung:** `s` là dãy con của `strs[i]` (chính nó) và không là dãy con của chuỗi khác ⇒ hợp lệ, nên đáp án ≥ `max len(s)`.
2. **(≤) Đáp án không thể lớn hơn:** giả sử đáp án thực là dãy con không chung `u` của `strs[i]`. Theo **bổ đề** ở mục 6, `strs[i] ∈ U` và `len(strs[i]) ≥ len(u)`. Vậy đáp án ≤ `max len(s)`.
3. **Trường hợp `-1`:** nếu `U` rỗng thì không tồn tại dãy con không chung nào (vì nếu có `u` thì `strs[i] ∈ U`, mâu thuẫn).
4. **So sánh theo chỉ số:** hai chuỗi trùng nội dung, mỗi chuỗi là dãy con của chuỗi kia ⇒ cả hai bị loại: đúng với định nghĩa ("không là dãy con của các chuỗi **còn lại**" — bản sao cũng là một chuỗi còn lại).
5. **`check` đúng (tham lam):** lấy khớp sớm nhất không bao giờ làm mất cơ hội khớp các ký tự sau — nếu tồn tại một cách khớp thì cách khớp "sớm nhất" cũng tồn tại.
6. **Cách 4 đúng:** chuỗi bị loại nếu có bản sao; ngược lại chỉ chuỗi **dài hơn** hoặc cùng độ dài-và-giống-hệt mới chứa được nó; vì không có bản sao nên chỉ chuỗi **dài hơn** (đứng trước trong `order`) cần kiểm tra. Duyệt từ dài đến ngắn nên ứng viên hợp lệ đầu tiên là cực đại.

Ta cũng đối chiếu thực nghiệm với oracle vét cạn (Cách 1) ở mục 12.

## 11. Phân tích Time / Space Complexity

`n = len(strs)`, `m` = độ dài lớn nhất của một chuỗi.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Mọi dãy con + kiểm tra | `O(n² · 2^m · m)` | `O(2^m · m)` | oracle, chậm |
| 2. `Counter` mọi dãy con | `O(n · 2^m · m)` | `O(n · 2^m · m)` | đơn giản, mũ theo `m` |
| 3. Chỉ thử chuỗi gốc | `O(n² · m)` | `O(1)` | **chính thức** |
| 4. Sắp xếp + dừng sớm | `O(n log n · m + n² · m)` | `O(n)` | thường nhanh hơn Cách 3 |

Với `n ≤ 50`, `m ≤ 10`: Cách 3 cần ≈ `50 · 50 · 10 = 25 000` phép — gần như tức thì. `check` là `O(m)` vì `j` chạy tối đa `len(t)` bước và `i` chỉ tăng.

## 12. Bộ test

Chiến lược:

1. **Ví dụ đề** (cả 4 cách + `Solution`).
2. **Edge case:** hai chuỗi giống hệt, chuỗi một ký tự, chuỗi là dãy con của chuỗi khác, nhiều chuỗi trùng, mọi chuỗi khác nhau hoàn toàn, chuỗi dài 10.
3. **Phản ví dụ cho các cách làm sai** (xem mục 13).
4. **Đối chiếu nhiều input** bằng `assert`: oracle vét cạn (Cách 1, kiểm tra bằng `iter`) với Cách 2, 3, 4 và `Solution` trên hàng nghìn input nhỏ có seed; thêm input đúng ràng buộc lớn (`n` tới 50, `m` tới 10) so Cách 2/3/4.
5. **Hiệu năng** trên ca xấu: `n = 50`, `m = 10`.

In [16]:
IMPLS = [lus_bruteforce, lus_counter, lus_candidates, lus_sorted, lambda strs: Solution().findLUSlength(strs)]
NAMES = ["bruteforce", "counter", "candidates", "sorted", "Solution"]

cases = [
    (["aba", "cdc", "eae"], 3),                 # ví dụ 1
    (["aaa", "aaa", "aa"], -1),                 # ví dụ 2
    (["aa", "aa"], -1),                         # hai chuỗi giống hệt
    (["a", "b"], 1),                            # một ký tự khác nhau
    (["a", "a"], -1),
    (["ab", "ba"], 2),                          # cùng bộ chữ, khác thứ tự
    (["a", "ab"], 2),                           # chuỗi ngắn là dãy con của chuỗi dài
    (["ab", "a"], 2),
    (["abc", "abc", "ab"], -1),                 # ngắn bị chứa, dài bị trùng
    (["abc", "ab", "a"], 3),
    (["aabbcc", "aabbcc", "cb"], 2),
    (["aabbcc", "aabbcc", "bc", "bcc", "aabbccc"], 7),
    (["a", "b", "c"], 1),
    (["abcdefghij", "abcdefghij", "abcdefghi"], -1),   # độ dài 10
    (["abcdefghij", "jihgfedcba"], 10),
    (["x", "x", "x", "y", "y"], -1),            # nhiều bản sao
    (["ab", "ab", "ac"], 2),                    # "ac" độc quyền (ab không chứa ac)
]
for strs, expected in cases:
    for f, name in zip(IMPLS, NAMES):
        got = f(strs)
        assert got == expected, (name, strs, got, expected)
print(len(cases), "ca cố định: cả 5 cách đều đúng")

17 ca cố định: cả 5 cách đều đúng


In [17]:
# Đối chiếu ngẫu nhiên, input NHỎ (có seed): oracle vét cạn vs mọi cách khác
rng = random.Random(522)
checked = 0
for _ in range(4000):
    n = rng.randint(2, 6)
    alpha = "abc"[: rng.randint(1, 3)]
    strs = ["".join(rng.choice(alpha) for _ in range(rng.randint(1, 5))) for _ in range(n)]
    expected = lus_bruteforce(strs)
    for f, name in zip(IMPLS[1:], NAMES[1:]):
        assert f(strs) == expected, (name, strs, f(strs), expected)
    checked += 1
print(checked, "input nhỏ ngẫu nhiên: mọi cách khớp oracle vét cạn")

# Đối chiếu input ĐÚNG RÀNG BUỘC LỚN (n tới 50, m tới 10, chữ thường): Cách 2 (hash) vs Cách 3, 4, Solution
for round_ in range(120):
    n = rng.randint(2, 50)
    alpha = "abcdefghijklmnopqrstuvwxyz"[: rng.choice([1, 2, 3, 5, 26])]
    strs = ["".join(rng.choice(alpha) for _ in range(rng.randint(1, 10))) for _ in range(n)]
    expected = lus_counter(strs)
    for f, name in zip(IMPLS[2:], NAMES[2:]):
        assert f(strs) == expected, (name, strs)
print("120 input theo đúng ràng buộc lớn: Cách 2 = Cách 3 = Cách 4 = Solution")

4000 input nhỏ ngẫu nhiên: mọi cách khớp oracle vét cạn


120 input theo đúng ràng buộc lớn: Cách 2 = Cách 3 = Cách 4 = Solution


In [18]:
# Kiểm chứng bổ đề mục 6 bằng thực nghiệm: đáp án theo dãy con == đáp án chỉ xét chuỗi gốc
rng = random.Random(99)
for _ in range(1500):
    n = rng.randint(2, 5)
    strs = ["".join(rng.choice("ab") for _ in range(rng.randint(1, 6))) for _ in range(n)]
    best_sub = lus_counter(strs)                      # xét mọi dãy con
    best_whole = max((len(s) for i, s in enumerate(strs)
                      if not any(is_subseq(s, t) for j, t in enumerate(strs) if j != i)), default=-1)
    assert best_sub == best_whole
print("bổ đề: OK (1500 input)")

# Hiệu năng ca xấu: n = 50, m = 10, bảng chữ nhỏ nên nhiều chuỗi bị chứa nhau
rng = random.Random(1)
strs = ["".join(rng.choice("ab") for _ in range(10)) for _ in range(50)]
for f, name in zip(IMPLS[1:], NAMES[1:]):
    t0 = time.perf_counter()
    r = f(strs)
    print(f"{name:<11} -> {r:>3}   {time.perf_counter() - t0:.4f}s")

bổ đề: OK (1500 input)
counter     ->  10   0.0028s
candidates  ->  10   0.0029s
sorted      ->  10   0.0000s
Solution    ->  10   0.0032s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: so sánh nội dung (`s != t`) thay vì chỉ số (`i != j`)

Muốn "bỏ qua chính nó" nhưng viết `if s != t and check(s, t)`. Khi có **hai chuỗi trùng nội dung**, mỗi bản bị bỏ qua so với bản kia ⇒ cả hai bị coi là độc quyền ⇒ trả sai.

In [19]:
def lus_wrong_equal(strs):
    def check(s, t):
        i = 0
        for ch in t:
            if i < len(s) and ch == s[i]:
                i += 1
        return i == len(s)

    ans = -1
    for s in strs:
        for t in strs:
            if s != t and check(s, t):          # SAI: bỏ qua mọi chuỗi cùng nội dung
                break
        else:
            ans = max(ans, len(s))
    return ans


print(lus_wrong_equal(["aaa", "aaa", "aa"]), "(đúng phải là -1)")
print(lus_wrong_equal(["aa", "aa"]), "(đúng phải là -1)")

3 (đúng phải là -1)
2 (đúng phải là -1)


**Vì sao sai:** hai bản `"aaa"` là **hai chuỗi khác nhau theo vị trí** trong mảng; mỗi bản là dãy con của bản kia. **Sửa:** so theo **chỉ số** `i != j` (hoặc đếm số lần xuất hiện bằng `Counter`).

### Lỗi 2: nhầm **dãy con** với **chuỗi con** (dùng `s in t`)

`"ac" in "abc"` là `False` vì `in` yêu cầu liền nhau. Nếu dùng `s in t` để kiểm tra, ta coi `s` là "không bị chứa" dù thật ra nó vẫn là dãy con. Hãy để máy **tìm một phản ví dụ nhỏ nhất** (có seed):

In [20]:
def lus_wrong_substring(strs):
    ans = -1
    for i, s in enumerate(strs):
        if not any(i != j and s in t for j, t in enumerate(strs)):     # SAI: chuỗi con, không phải dãy con
            ans = max(ans, len(s))
    return ans


rng = random.Random(5)
found = None
for _ in range(200000):
    n = rng.randint(2, 3)
    strs = ["".join(rng.choice("ab") for _ in range(rng.randint(1, 4))) for _ in range(n)]
    if lus_wrong_substring(strs) != lus_candidates(strs):
        if found is None or sum(map(len, strs)) < sum(map(len, found)):
            found = strs
print("phản ví dụ:", found)
print("đáp án sai (dùng `in`):", lus_wrong_substring(found), "| đáp án đúng:", lus_candidates(found))

phản ví dụ: ['bab', 'bb', 'bab']
đáp án sai (dùng `in`): 2 | đáp án đúng: -1


**Vì sao sai:** trong phản ví dụ trên, một chuỗi là dãy con (bỏ vài ký tự) của chuỗi kia nhưng **không liền nhau**, nên `in` báo "không chứa" và ta đếm nhầm. **Sửa:** dùng hàm hai con trỏ `check(s, t)`.

### Lỗi 3: chỉ loại chuỗi trùng, quên kiểm tra "bị chứa trong chuỗi dài hơn"

Ý nghĩ: "chuỗi nào xuất hiện **đúng một lần** thì độc quyền, lấy `max` độ dài". Sai vì chuỗi duy nhất vẫn có thể là dãy con của chuỗi khác.

In [21]:
def lus_wrong_counter_only(strs):
    cnt = Counter(strs)
    return max((len(s) for s in strs if cnt[s] == 1), default=-1)    # SAI: thiếu kiểm tra dãy con


print(lus_wrong_counter_only(["abc", "abc", "ab"]), "(đúng phải là -1: 'ab' nằm trong 'abc')")
print(lus_wrong_counter_only(["aaa", "aa"]), "(đúng: 3, vì 'aaa' dài hơn)")
print(lus_wrong_counter_only(["aaaa", "aaa", "aaa", "aa"]), "(đúng: 4)")

2 (đúng phải là -1: 'ab' nằm trong 'abc')
3 (đúng: 3, vì 'aaa' dài hơn)
4 (đúng: 4)


**Vì sao sai:** ở ví dụ đầu `"ab"` chỉ xuất hiện một lần nhưng là dãy con của `"abc"`. **Sửa:** sau khi loại chuỗi có bản sao, **vẫn phải** kiểm tra dãy con với các chuỗi còn lại (Cách 4).

### Lỗi 4: kiểm tra ngược chiều (`t` là dãy con của `s`)

Đề hỏi `s` có bị chứa trong `t` không, tức `check(s, t)`: "**s là dãy con của t**". Nhầm thành `check(t, s)` làm kết quả đảo ngược. Ví dụ `["abc", "ab"]`: đáp án đúng `3`.

In [22]:
def lus_wrong_direction(strs):
    ans = -1
    for i, s in enumerate(strs):
        for j, t in enumerate(strs):
            if i != j and is_subseq(t, s):          # SAI: đảo chiều
                break
        else:
            ans = max(ans, len(s))
    return ans


print(lus_wrong_direction(["abc", "ab"]), "(đúng: 3)")
print(lus_candidates(["abc", "ab"]))

2 (đúng: 3)
3


### Lỗi 5: khởi tạo `ans = 0` thay vì `-1`

Khi không có dãy con độc quyền, đề yêu cầu trả **`-1`**. Nếu khởi tạo `ans = 0` và dùng `max`, ta trả `0` cho ví dụ 2.

In [23]:
def lus_wrong_init(strs):
    ans = 0                                     # SAI: phải là -1
    for i, s in enumerate(strs):
        for j, t in enumerate(strs):
            if i != j and is_subseq(s, t):
                break
        else:
            ans = max(ans, len(s))
    return ans


print(lus_wrong_init(["aaa", "aaa", "aa"]), "(đúng: -1)")
print(lus_candidates(["aaa", "aaa", "aa"]))

0 (đúng: -1)
-1


**Checklist chống lỗi:**

- [ ] So sánh theo **chỉ số** (`i != j`), không theo nội dung.
- [ ] Dùng **dãy con** (hai con trỏ), không phải `in`.
- [ ] Với chuỗi duy nhất vẫn phải kiểm tra bị chứa bởi chuỗi dài hơn.
- [ ] Hướng kiểm tra: `check(s, t)` = "`s` là dãy con của `t`".
- [ ] `ans` bắt đầu bằng `-1`.

## 14. Mẹo & nhận dạng pattern

- **"Dãy con độc quyền/không chung"** + ràng buộc nhỏ ⇒ nghĩ ngay **bổ đề "thử chính các chuỗi gốc"**: dãy con dài nhất thoả tính chất phản-bắc cầu (nếu `u ⊂ s ⊂ t` thì `u ⊂ t`) luôn là một chuỗi nguyên bản.
- **Tính chất bắc cầu của quan hệ "là dãy con"** là chìa khoá: nếu `s` bị chứa trong `t` thì mọi dãy con của `s` cũng bị chứa trong `t`.
- **Chuỗi trùng:** hai chuỗi giống hệt luôn chứa nhau. Xử lý bằng chỉ số hoặc `Counter`.
- **Kiểm tra dãy con** = mẫu hai con trỏ `O(len(t))`; học thuộc: `for ch in t: if i < len(s) and ch == s[i]: i += 1`.
- **Sắp xếp giảm dần theo độ dài** để "dừng ngay khi gặp ứng viên đầu tiên" là mẹo chung cho bài "tối đa hoá" có thể kiểm tra ứng viên độc lập.
- **`for ... else`** gọn cho dạng "không có phần tử nào vi phạm".
- Bài họ hàng: 521 (Longest Uncommon Subsequence I), 392 (Is Subsequence), 1143 (Longest Common Subsequence), 524 (Longest Word in Dictionary through Deleting).

## 15. Biến thể

1. **Bài 521 — Longest Uncommon Subsequence I** (đúng **hai** chuỗi `a`, `b`): nếu `a == b` thì `-1`, ngược lại `max(len(a), len(b))` (vì chuỗi dài hơn không thể là dãy con của chuỗi ngắn hơn; nếu bằng độ dài nhưng khác nhau thì không chứa nhau).
2. **Trả về chính chuỗi** độc quyền dài nhất (nếu nhiều, lấy chuỗi nhỏ nhất theo thứ tự từ điển).
3. **Đếm** có bao nhiêu chuỗi (theo chỉ số) là độc quyền.
4. **Độ dài không giới hạn nhỏ:** vì Cách 3 chỉ `O(n² · m)` nên vẫn dùng được khi `m` lớn (Cách 1, 2 thì không).
5. **"Dãy con chung dài nhất"** (LCS, bài 1143) là bài quy hoạch động khác hẳn — nhưng cùng khái niệm dãy con.

Minh hoạ nhanh biến thể 1, 2, 3:

In [24]:
def lus_one(a, b):
    """Bài 521: hai chuỗi."""
    return -1 if a == b else max(len(a), len(b))


def uncommon_indices(strs):
    """Biến thể 3: chỉ số các chuỗi độc quyền."""
    return [i for i, s in enumerate(strs)
            if not any(i != j and is_subseq(s, t) for j, t in enumerate(strs))]


def longest_uncommon_string(strs):
    """Biến thể 2: chuỗi độc quyền dài nhất (nhỏ nhất theo từ điển nếu nhiều), hoặc None."""
    cands = [strs[i] for i in uncommon_indices(strs)]
    if not cands:
        return None
    return min(cands, key=lambda s: (-len(s), s))


print(lus_one("aba", "cdc"), lus_one("aaa", "bbb"), lus_one("aaa", "aaa"))        # 3 3 -1
print(uncommon_indices(["aabbcc", "aabbcc", "cb"]))                                # [2]
print(longest_uncommon_string(["ab", "ba", "a"]), longest_uncommon_string(["aaa", "aaa", "aa"]))

3 3 -1
[2]
ab None


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi dãy con + kiểm tra | định nghĩa đề, vét cạn | `O(n² · 2^m · m)` | `O(2^m · m)` | chỉ làm **oracle** để test |
| 2. `Counter` mọi dãy con | đếm số chuỗi chứa mỗi dãy con, lấy `count == 1` | `O(n · 2^m · m)` | `O(n · 2^m · m)` | `m` nhỏ (≤ ~15), muốn code đơn giản không cần bổ đề |
| 3. Chuỗi gốc + hai con trỏ | bổ đề: chỉ thử `n` chuỗi gốc | `O(n² · m)` | `O(1)` | **mặc định khi nộp bài** |
| 4. Sắp xếp + dừng sớm | duyệt từ dài đến ngắn, bỏ chuỗi trùng | `O(n log n · m + n² · m)` | `O(n)` | muốn dừng sớm, dữ liệu có chuỗi dài độc quyền |

## 17. Bài tập vận dụng (dễ → khó)

Mỗi bài có cell TODO để bạn tự làm (chưa gọi hàm; chỉ gán `None` hoặc `def` rỗng). Đáp án ở mục 19, gợi ý ở mục 18.

### Bài 1 (đoán output)

Dự đoán giá trị của các biểu thức sau (ghi vào list `bai1_du_doan` theo đúng thứ tự):

1. `is_subseq("ace", "abcde")`
2. `is_subseq("aec", "abcde")`
3. `is_subseq("", "xyz")`
4. `Solution().findLUSlength(["abc", "ab", "a"])`
5. `Solution().findLUSlength(["ab", "ab", "b"])`

In [25]:
# TODO Bài 1: điền 5 giá trị dự đoán, ví dụ [True, False, ...]
bai1_du_doan = None

### Bài 2 (tính tay)

(a) Chuỗi `"abcd"` (bốn chữ khác nhau) có bao nhiêu dãy con **khác nhau** (kể cả rỗng)? (b) Chuỗi `"aaaa"` có bao nhiêu dãy con **khác nhau**? (c) `strs = ["abc", "abd", "ab"]`: đáp án là gì? Ghi cả ba kết quả (số, số, số) vào `bai2`.

In [26]:
# TODO Bài 2: bai2 = (a, b, c)
bai2 = None

### Bài 3 (điền TODO)

Hoàn thành hàm `is_subsequence(s, t)` bằng hai con trỏ (trả `True` nếu `s` là dãy con của `t`).

In [27]:
def is_subsequence(s, t):
    # TODO: i = 0; duyệt từng ký tự ch của t; nếu i < len(s) và ch == s[i] thì i += 1; cuối cùng so i với len(s)
    pass

### Bài 4 (tự viết lại)

Không nhìn mục 9, hãy tự viết hàm `lus(strs)` giải bài (Cách 3: thử từng chuỗi gốc, so theo chỉ số, `for ... else`).

In [28]:
# TODO Bài 4: tự viết lại
def lus(strs):
    pass

### Bài 5 (biến thể / mini challenge)

Viết hàm `containing_counts(strs)` trả về, **cho mỗi chuỗi**, số chuỗi **khác** (theo chỉ số) chứa nó làm dãy con — tức list `res` với `res[i] = số j != i sao cho strs[i] là dãy con của strs[j]`. Sau đó suy ra: `strs[i]` độc quyền ⇔ `res[i] == 0`. Dùng nó để viết lại đáp án bài 522 trong một dòng.

In [29]:
# TODO Bài 5
def containing_counts(strs):
    pass

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `is_subseq(s, t)` hỏi "các ký tự của `s` có xuất hiện lần lượt trong `t` không".
- Mức 2: `"aec"`: `a` ở vị trí 0, `e` ở vị trí 4 nhưng sau đó `c` không còn.
- Mức 3: dòng 4: `"abc"` không bị chứa trong `"ab"`/`"a"` ⇒ 3; dòng 5: `"ab"` trùng, `"b"` là dãy con của `"ab"` ⇒ -1.

**Bài 2**
- Mức 1: mỗi ký tự khác nhau cho "giữ/bỏ" ⇒ `2^4`.
- Mức 2: `"aaaa"`: dãy con khác nhau chỉ phụ thuộc số chữ `a` giữ lại (0..4).
- Mức 3: (a) 16, (b) 5. (c): `"abc"` có bị chứa trong `"abd"`/`"ab"` không? Không ⇒ độc quyền, dài 3 ⇒ 3 (cũng `"abd"`).

**Bài 3**
- Mức 1: một biến đếm `i` cho `s`.
- Mức 2: `for ch in t:` và điều kiện `i < len(s) and ch == s[i]`.
- Mức 3: `return i == len(s)`.

**Bài 4**
- Mức 1: hai vòng `for` lồng nhau với `enumerate`.
- Mức 2: `if i != j and is_subseq(s, t): break` và `else:` của vòng trong.
- Mức 3: `ans = max(ans, len(s))` trong khối `else`; `ans` bắt đầu `-1`.

**Bài 5**
- Mức 1: list comprehension, phần tử `i` là đếm các `j` thoả điều kiện.
- Mức 2: `sum(1 for j, t in enumerate(strs) if i != j and is_subseq(s, t))`.
- Mức 3: đáp án 522: `max((len(s) for s, c in zip(strs, res) if c == 0), default=-1)`.

## 19. Đáp án tham khảo

Chạy kiểm tra từng đáp án bằng `assert`.

In [30]:
# Bài 1
dap_an_1 = [is_subseq("ace", "abcde"), is_subseq("aec", "abcde"), is_subseq("", "xyz"),
            Solution().findLUSlength(["abc", "ab", "a"]), Solution().findLUSlength(["ab", "ab", "b"])]
print(dap_an_1)
assert dap_an_1 == [True, False, True, 3, -1]

# Bài 2
assert len(subsequence_set("abcd")) == 16 and len(subsequence_set("aaaa")) == 5
assert Solution().findLUSlength(["abc", "abd", "ab"]) == 3
bai2_dap_an = (16, 5, 3)
print(bai2_dap_an)

[True, False, True, 3, -1]
(16, 5, 3)


In [31]:
# Bài 3
def is_subsequence(s, t):
    i = 0
    for ch in t:
        if i < len(s) and ch == s[i]:
            i += 1
    return i == len(s)


rng = random.Random(3)
for _ in range(2000):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(0, 5)))
    t = "".join(rng.choice("abc") for _ in range(rng.randint(0, 8)))
    assert is_subsequence(s, t) == is_subseq_iter(s, t) == (s in set(subsequence_set(t)))
print("Bài 3: OK")

# Bài 4
def lus(strs):
    ans = -1
    for i, s in enumerate(strs):
        for j, t in enumerate(strs):
            if i != j and is_subsequence(s, t):
                break
        else:
            ans = max(ans, len(s))
    return ans


for strs, expected in cases:
    assert lus(strs) == expected
rng = random.Random(4)
for _ in range(1000):
    strs = ["".join(rng.choice("ab") for _ in range(rng.randint(1, 5))) for _ in range(rng.randint(2, 5))]
    assert lus(strs) == lus_bruteforce(strs)
print("Bài 4: OK")

Bài 3: OK
Bài 4: OK


In [32]:
# Bài 5
def containing_counts(strs):
    return [sum(1 for j, t in enumerate(strs) if i != j and is_subseq(s, t)) for i, s in enumerate(strs)]


def lus_from_counts(strs):
    res = containing_counts(strs)
    return max((len(s) for s, c in zip(strs, res) if c == 0), default=-1)


assert containing_counts(["aabbcc", "aabbcc", "cb"]) == [1, 1, 0]
assert containing_counts(["abc", "ab", "a"]) == [0, 1, 2]
for strs, expected in cases:
    assert lus_from_counts(strs) == expected
rng = random.Random(5)
for _ in range(1000):
    strs = ["".join(rng.choice("abc") for _ in range(rng.randint(1, 6))) for _ in range(rng.randint(2, 6))]
    assert lus_from_counts(strs) == lus_bruteforce(strs)
print("Bài 5: OK")

Bài 5: OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "ứng viên là chuỗi gốc, loại nếu bị chứa":**

```python
class Solution:
    def findLUSlength(self, strs: List[str]) -> int:
        def check(s: str, t: str) -> bool:      # s có là dãy con của t không
            i = j = 0
            while i < len(s) and j < len(t):
                if s[i] == t[j]:
                    i += 1
                j += 1
            return i == len(s)

        ans = -1
        for i, s in enumerate(strs):
            for j, t in enumerate(strs):
                if i != j and check(s, t):      # so theo CHỈ SỐ
                    break
            else:                                # không bị chứa -> độc quyền
                ans = max(ans, len(s))
        return ans
```

**Checklist trước khi submit:**

- [ ] Hiểu bổ đề: đáp án luôn là độ dài của một **chuỗi gốc** (nếu `u ⊂ s` và `s ⊂ t` thì `u ⊂ t`)?
- [ ] `check(s, t)` đúng chiều ("`s` là dãy con của `t`") và dùng hai con trỏ, không dùng `in`?
- [ ] So sánh bằng chỉ số `i != j` để xử lý đúng chuỗi trùng?
- [ ] `ans` khởi tạo `-1`; dùng `for ... else` hoặc cờ `ok` đúng chỗ?
- [ ] Test: hai chuỗi giống hệt, chuỗi là dãy con của chuỗi khác, toàn chuỗi khác nhau, độ dài 10, ví dụ đề?
- [ ] Đã đối chiếu với oracle vét cạn trên input nhỏ ngẫu nhiên có seed?